# Install & Import

In [1]:
!pip install -q sentence-transformers faiss-gpu-cu12 rank_bm25 transformers sacremoses 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 19.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 35.0 MB/s eta 0:00:00


In [2]:
import os
import pandas as pd
import numpy as np
import faiss
import pickle
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

# load dataset

In [3]:
dirname= "/kaggle/input/competitions/llm-agentic-legal-information-retrieval"

# train = pd.read_csv(dirname+"/train.csv")
# val = pd.read_csv(dirname+"/val.csv")
# test = pd.read_csv(dirname+"/test.csv")

laws_de = pd.read_csv(dirname+"/laws_de.csv")
court_considerations = pd.read_csv(dirname+"/court_considerations.csv")

In [4]:
print(f'''"\nlaws_de:"{laws_de.shape},
"\ncourt_considerations:"{court_considerations.shape}''')

"
laws_de:"(175933, 3),
"
court_considerations:"(2476315, 2)


In [5]:
laws_de.head()

,citation,text,title
0,Art. 1 112,Die Einwohnergemeinde Bern tritt der Schweizer...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
1,Art. 2 112,Die Einwohnergemeinde Bern wird ferner der Sch...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
2,Art. 3 Abs. 1 112,1 Falls die Schweizerische Eidgenossenschaft z...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
3,Art. 3 Abs. 2 112,2 Durch Anlage des neuen Verwaltungsgebäudes a...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
4,Art. 4 Abs. 1 112,1 Die Einwohnergemeinde Bern übernimmt im fern...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...


In [6]:
court_considerations.head()

,citation,text
0,BGE 139 I 2 E. 1.12.2011,betr. Verweigerung der Beiladung seien gutzuhe...
1,BGE 139 I 2 E. 2,Eventualiter sei die Rückweisung an die Vorins...
2,BGE 139 I 2 E. 5.1,"In der Sache ist vorweg zu prüfen, ob der Ents..."
3,BGE 139 I 2 E. 5.2,Art. 34 Abs. 1 BV gewährleistet in allgemeiner...
4,BGE 139 I 2 E. 5.3,Im vorliegenden Fall geht es nicht um die Gült...


laws_de is like law book

court_considerations is like case studies

| laws_de          | court_considerations |
| ---------------- | -------------------- |
| Rules            | Interpretations      |
| Short text       | Long text            |
| Structured       | Complex              |
| Easier retrieval | Harder retrieval     |


# Prepare Corpus

In [7]:
laws_de.columns

Index(['citation', 'text', 'title'], dtype='object')

In [8]:
#prepare corpus
def prepare(df):
    df["title"]=df.get("title","").fillna("")
    df["text"]=df["text"].fillna("")
    df["full_text"]=df["title"]*3+" "+df["text"]
    return df

laws_corpus=prepare(laws_de)

laws_docs = laws_corpus["full_text"].tolist()
laws_ids = laws_corpus["citation"].tolist()

court_docs=court_considerations["text"].tolist()
court_ids=court_considerations["citation"].tolist()

all_docs=laws_docs+court_docs
all_ids=laws_ids+court_ids


id2text =dict(zip(all_ids,all_docs))

del laws_docs, laws_ids, court_docs, court_ids, laws_corpus, laws_de, court_considerations

In [9]:
# !rm -rf /kaggle/working/cache

In [10]:
CACHE_DIR = "/kaggle/working/cache"
os.makedirs(CACHE_DIR, exist_ok=True)


# np.save(CACHE_DIR+"/all_ids.npy", np.array(all_ids))

# with open(CACHE_DIR+"/all_docs.pkl","wb") as f:
#     pickle.dump(all_docs,f)

# with open(CACHE_DIR+"/id2text.pkl","wb") as f:
#     pickle.dump(id2text,f)

# Model Initialization

In [11]:
# cross Lingual
model_path= "/kaggle/input/models/yethukmutt/bge-m3/transformers/m3/1/bge-m3"
reranker_path="/kaggle/input/models/andreasbis/baai-bge-reranker-v2-m3/transformers/default/1"

embed_model =SentenceTransformer(model_path).to("cuda") # best choice
embed_model._first_module().auto_model.half()

reranker = CrossEncoder(reranker_path)
reranker.model.to("cuda")
reranker.model.half()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


# Translation Batch Precompute

In [ ]:
from transformers import MarianMTModel, MarianTokenizer, MarianConfig

translate_path="/kaggle/input/models/aaryanverma/helsinki-nlpopus-mt-en-de/transformers/default/1/opus-mt-en-de"
tokenizer=MarianTokenizer.from_pretrained(translate_path)
translator=MarianMTModel.from_pretrained(translate_path).to("cuda")

In [ ]:
#Translation;Batch Precompute
# from tqdm import tqdm
# test = pd.read_csv(dirname+"/test.csv")

def translate(query):
    inputs=tokenizer(query, return_tensors='pt',trancation=True, padding=True).to("cuda")
    outputs=translator.generate(**inputs,max_length=256)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# all_translated={}

# for q in tqdm(test['query'],desc="Translating"):
#     all_translated[q]=translate(q)


# LOAD OR BUILD EMBEDDINGS


In [ ]:
print(faiss.get_num_gpus())

In [ ]:
# import gc, torch
# gc.collect()
# torch.cuda.empty_cache()

### CHUNKED EMBEDDING (WITH CHECKPOINTS)

In [ ]:
import torch
import gc

torch.set_grad_enabled(False)

CHUNK_SIZE = 50000
emb_dir = CACHE_DIR + "/emb_chunks"
os.makedirs(emb_dir, exist_ok=True)

embed_model.max_seq_length = 256   # 🔥 CRITICAL FIX

num_chunks = (len(all_docs) + CHUNK_SIZE - 1) // CHUNK_SIZE

for i in range(num_chunks):
    chunk_path = f"{emb_dir}/chunk_{i}.npy"

    if os.path.exists(chunk_path):
        print(f"✅ Skipping chunk {i}")
        continue

    start = i * CHUNK_SIZE
    end = min((i + 1) * CHUNK_SIZE, len(all_docs))

    chunk_docs = all_docs[start:end]

    emb = embed_model.encode(
        chunk_docs,
        batch_size=16,          # 🔥 FIXED (was 128)
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True
    ).astype("float32")

    np.save(chunk_path, emb)

    # 🔥 memory cleanup
    del emb
    torch.cuda.empty_cache()
    gc.collect()

    print(f"💾 Saved chunk {i}")

In [ ]:
nvidia-smi


# MERGE EMBEDDINGS

In [ ]:
chunks = []
for i in range(num_chunks):
    chunk_path = f"{emb_dir}/chunk_{i}.npy"
    chunks.append(np.load(chunk_path))

embeddings = np.vstack(chunks)

np.save(CACHE_DIR + "/embeddings.npy", embeddings)

print("✅ All embeddings merged")

# FAISS index

In [22]:
#faiss index; law (full)
import faiss
# embeddings=embed_model.encode(
#     all_docs,
#     batch_size=8,
#     normalize_embeddings=True,
#     convert_to_numpy=True,
#     show_progress_bar= True,
# ).astype("float32")

dim =embeddings.shape[1]
quantizer=faiss.IndexFlatIP()
index=faiss.IndexIVFFlat(
    quantizer,
    dim,
    2048,
    faiss.METRIC_INNER_PRODUCT
)

index.train(embeddings[:200000])
index.add(embeddings)
index.nprobe=60


faiss.write_index(index, CACHE_DIR+"/faiss.index")

Batches:   0%|          | 0/331531 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 79.81 MiB is free. Including non-PyTorch memory, this process has 14.48 GiB memory in use. Of the allocated memory 14.19 GiB is allocated by PyTorch, and 181.74 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# BM25

In [ ]:
def tokenize(text):
    t = t.lower()
    t = re.sub(r"[^a-z0-9§. ]", " ", t)
    return t.split()

bm25=BM25Okapi([token(t) for t in all_docs])


with open(CACHE_DIR+"/bm25.pkl","wb") as f:
    pickle.dump(dm25,f)

# Loads from cache

In [17]:

# emb_path = f"{CACHE_DIR}/embeddings.npy"
# id_path = f"{CACHE_DIR}/ids.npy"
# index_path = f"{CACHE_DIR}/faiss.index"
# bm_path = f"{CACHE_DIR}/bm25.pkl"

# def tokenize(t):
#     t = t.lower()
#     t = re.sub(r"[^a-z0-9§. ]", " ", t)
#     return t.split()

# if os.path.exists(index_path):

#     print("🚀 Loading cached artifacts...")

#     embeddings = np.load(emb_path)
#     all_ids = np.load(id_path).tolist()
#     id2text = id2text

#     index = faiss.read_index(index_path)

#     with open(bm_path, "rb") as f:
#         all_docs = pickle.load(f)

# else:
#     print("⚙️ Building embeddings + index...")

#     embeddings = embed_model.encode(
#         all_docs,
#         batch_size=16,
#         normalize_embeddings=True,
#         convert_to_numpy=True,
#         show_progress_bar=True
#     ).astype("float32")

#     dim = embeddings.shape[1]

#     quantizer = faiss.IndexFlatIP(dim)

#     index = faiss.IndexIVFFlat(
#         quantizer,
#         dim,
#         2048,
#         faiss.METRIC_INNER_PRODUCT
#     )

#     index.train(embeddings[:200000])
#     index.add(embeddings)
#     index.nprobe = 60

#     faiss.write_index(index, index_path)
#     np.save(emb_path, embeddings)
#     np.save(id_path, np.array(all_ids))

#     with open(docs_path, "wb") as f:
#         pickle.dump(all_docs, f)

#     print("✅ Saved all artifacts")


# For inference and tunning

In [ ]:
import numpy as np
import faiss
import pickle
import os

CACHE_DIR = "/kaggle/working/cache"

embeddings = np.load(CACHE_DIR + "/embeddings.npy")
all_ids = np.load(CACHE_DIR + "/all_ids.npy").tolist()

index = faiss.read_index(CACHE_DIR + "/faiss.index")

with open(CACHE_DIR + "/bm25.pkl", "rb") as f:
    bm25 = pickle.load(f)

with open(CACHE_DIR + "/all_docs.pkl", "rb") as f:
    all_docs = pickle.load(f)

with open(CACHE_DIR + "/id2text.pkl", "rb") as f:
    id2text = pickle.load(f)

print("🚀 CACHE LOADED — READY FOR TUNING")

# Intent detection

In [ ]:
def detect_intent(query):
    q = query.lower()

    l = sum(k in q for k in ["article", "art.", "section", "law"])
    c = sum(k in q for k in ["court", "case", "decision", "judgment"])

    if l > c:
        return "law"
    elif c > l:
        return "court"
    return "mixed"

In [ ]:
def vector_search(q, intent):
    emb = embed_model.encode([q], normalize_embeddings=True)

    if intent == "law":
        lk, ck = 40, 10
    elif intent == "court":
        lk, ck = 10, 40
    else:
        lk, ck = 25, 25

    ls, li = law_index.search(emb, lk)
    cs, ci = court_index.search(emb, ck)

    law_res = [(law_ids[i], law_docs[i], s) for i, s in zip(li[0], ls[0])]
    court_res = [(court_ids[i], court_docs[i], s) for i, s in zip(ci[0], cs[0])]

    return law_res + court_res

In [ ]:
def bm25_search(q, k=30):
    scores = bm25.get_scores(tokenize(q))
    idx = np.argsort(scores)[::-1][:k]
    return [(law_ids[i], law_docs[i], scores[i]) for i in idx]

In [ ]:
def dedup(results):
    best = {}
    for doc_id, text, score in results:
        if doc_id not in best or best[doc_id] < score:
            best[doc_id] = score
    return [(k, "", v) for k, v in best.items()]

In [ ]:
def get_weights(intent):
    if intent == "law":
        return {"vector": 1.0, "bm25": 1.2}
    elif intent == "court":
        return {"vector": 1.2, "bm25": 0.5}
    return {"vector": 1.0, "bm25": 1.0}


def weighted_rrf(res, weights, k=60):
    scores = {}
    for key, results in res.items():
        w = weights.get(key, 1.0)
        for rank, (doc_id, _, _) in enumerate(results):
            scores[doc_id] = scores.get(doc_id, 0) + w*(1/(k+rank+1))
    return scores

In [ ]:
def pattern_boost(scores, q):
    if re.search(r"art\.?\s?\d+", q.lower()):
        return {k: v*1.2 if "Art." in k else v for k,v in scores.items()}
    if "bge" in q.lower():
        return {k: v*1.2 if "BGE" in k else v for k,v in scores.items()}
    return scores

In [ ]:
def dynamic_top_k(scores):
    return min(20, max(8, int(len(scores)*0.3)))

In [ ]:
def rerank(query, docs):
    pairs = [[query, f"{doc_id} {text}"] for doc_id, text in docs]

    scores = []
    for i in range(0, len(pairs), 32):
        scores.extend(reranker.compute_score(pairs[i:i+32], normalize=True))

    ranked = sorted(zip(scores, docs), reverse=True)

    k = dynamic_top_k([s for s,_ in ranked])
    return [doc_id for _, (doc_id, _) in ranked[:k]]

In [ ]:
def pipeline(query):
    
    intent = detect_intent(query)

    q_en = query
    q_de = all_translated.get(query, query)

    vec_en = vector_search(q_en, intent)
    vec_de = vector_search(q_de, intent)

    vec = dedup(vec_en + vec_de)

    bm = dedup(bm25_search(q_de))

    fused = weighted_rrf({
        "vector": vec,
        "bm25": bm
    }, get_weights(intent))

    fused = pattern_boost(fused, query)

    top_ids = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:60]

    docs = [(doc_id, id2text[doc_id]) for doc_id,_ in top_ids]

    final = rerank(query, docs)

    return ";".join(final)

In [ ]:
preds = [pipeline(q) for q in tqdm(test["query"])]

sub = pd.DataFrame({
    "query_id": test["query_id"],
    "predicted_citations": preds
})

sub.to_csv("submission.csv", index=False)

In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
